# Lab 1: Bayesian Optimization

**Student:** Lopatenko George (334771)  
**Course:** Automated Machine Learning

We compare manual Bayesian optimization, manual random search, Optuna TPE,
and Optuna RandomSampler for tuning histogram gradient boosting on MAGIC Gamma
Telescope. The objective is mean cross-validation ROC AUC under equal trial budgets.

## 0. Setup


In [1]:
import shutil
from pathlib import Path
from time import perf_counter

import kaleido
import numpy as np
import optuna
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
from IPython.display import Image, Markdown, display
from plotly.subplots import make_subplots
from prettytable import PrettyTable
from scipy.stats import norm
from sklearn.datasets import fetch_openml
from sklearn.decomposition import PCA
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestRegressor
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern
from sklearn.inspection import permutation_importance
from sklearn.metrics import r2_score, roc_auc_score, roc_curve
from sklearn.model_selection import KFold, StratifiedKFold, cross_val_score, train_test_split
from threadpoolctl import threadpool_limits

SEED = 334771
SEARCH_SEEDS = [SEED, SEED + 1, SEED + 2]
N_TRIALS, N_INITIAL, N_FOLDS = 40, 8, 3
METHODS = ['Manual BO', 'Manual random', 'Optuna TPE', 'Optuna random']
COLORS = dict(zip(METHODS, ['#0072B2', '#D55E00', '#009E73', '#CC79A7']))
DASHES = dict(zip(METHODS, ['solid', 'dash', 'dot', 'dashdot']))
SYMBOLS = dict(zip(METHODS, ['circle', 'square', 'diamond', 'triangle-up']))
FIG_WIDTH, FIG_SCALE = 1400, 3
LAB_DIR = Path.cwd()
if (LAB_DIR / 'automl/lab-1').is_dir():
    LAB_DIR = LAB_DIR / 'automl/lab-1'
RESULTS_DIR, FIGURES_DIR = LAB_DIR / 'results', LAB_DIR / 'figures'
for folder in [RESULTS_DIR, FIGURES_DIR]:
    folder.mkdir(parents=True, exist_ok=True)
browser_paths = [shutil.which(name) for name in
                 ['google-chrome', 'chromium', 'chromium-browser', 'chrome']]
browser_paths += ['/Applications/Google Chrome.app/Contents/MacOS/Google Chrome']
if not any(path and Path(path).exists() for path in browser_paths):
    import os
    os.environ['BROWSER_PATH'] = str(kaleido.get_chrome_sync())

optuna.logging.set_verbosity(optuna.logging.WARNING)
pio.templates.default = 'plotly_white'

/Users/lopatenko/Desktop/itmo/itmo-ml-2025/automl/lab-1/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def show_table(frame: pd.DataFrame, caption: str, precision: int = 4) -> None:
    def format_value(value) -> str:
        if pd.isna(value):
            return '—'
        if isinstance(value, (float, np.floating)):
            return f'{value:.{precision}f}'
        return str(value)

    table = PrettyTable()
    table.title = caption
    table.field_names = [str(column) for column in frame.columns]
    table.align = 'l'
    for field in table.field_names:
        table.max_width[field] = 32
    for row in frame.itertuples(index=False, name=None):
        table.add_row([format_value(value) for value in row])
    print(table)


def show_and_save(fig: go.Figure, stem: str, height: int = 780) -> Path:
    fig.update_layout(
        template='plotly_white', width=FIG_WIDTH, height=height,
        font=dict(family='Arial', size=17, color='#243447'),
        margin=dict(l=85, r=55, t=125, b=170),
        title_y=0.94, title_yanchor='top',
        legend=dict(orientation='h', y=-80 / (height - 295), x=0, yanchor='top'),
    )
    fig.add_annotation(
        text='Source: UCI MAGIC Gamma Telescope / OpenML 1120; experiments in this notebook',
        x=0, y=0, yshift=-135, xref='paper', yref='paper', showarrow=False,
        xanchor='left', yanchor='top', font=dict(size=12, color='#536170'),
    )
    fig.update_yaxes(automargin=True)
    path = FIGURES_DIR / f'{stem}.png'
    fig.write_image(str(path), scale=FIG_SCALE, width=FIG_WIDTH, height=height)
    fig.write_html(FIGURES_DIR / f'{stem}.html', include_plotlyjs=True)
    fig.write_json(FIGURES_DIR / f'{stem}.json')
    fig.show(config={'displaylogo': False, 'responsive': True})
    # display(Image(filename=str(path), width=FIG_WIDTH))
    return path

## 1. Dataset and exploratory analysis

[MAGIC Gamma Telescope](https://archive.ics.uci.edu/dataset/159/magic%2Bgamma%2Btelescope)
contains 19,020 simulated events and ten numerical image descriptors.
The target is gamma signal (`g`, encoded as 1) versus hadronic background
(`h`, encoded as 0). The data are loaded from **OpenML ID 1120, version 1**;
the row identifier is excluded from the predictors.

The simulated class proportions differ from real telescope observations.
The dataset documentation recommends ROC analysis rather than accuracy.
We optimize ROC AUC and additionally inspect the low-false-positive-rate region.

Identical feature rows are deduplicated before a stratified 80/20 train/test split.
Conflicting labels on identical rows are checked before removal. Three fixed
stratified cross-validation folds are constructed within the training partition.

In [3]:
dataset = fetch_openml(data_id=1120, as_frame=True,
                       data_home=str(LAB_DIR / '.cache/sklearn'))
X_raw = dataset.data.copy()
X_raw.columns = X_raw.columns.str.rstrip(':')
y_raw = (dataset.target.astype(str) == 'g').astype(int).rename('gamma')
assert set(dataset.target.astype(str)) == {'g', 'h'}
assert X_raw.shape == (19020, 10)
assert 'ID' not in X_raw.columns
assert np.isfinite(X_raw.to_numpy()).all()
combined = X_raw.assign(gamma=y_raw)
assert combined.groupby(list(X_raw.columns), dropna=False).gamma.nunique().max() == 1
keep = ~X_raw.duplicated()
X, y = X_raw.loc[keep].copy(), y_raw.loc[keep].copy()
train_ids, test_ids = train_test_split(
    X.index.to_numpy(), test_size=0.20, stratify=y, random_state=SEED,
)
X_train, X_test = X.loc[train_ids], X.loc[test_ids]
y_train, y_test = y.loc[train_ids], y.loc[test_ids]
CV_SPLITS = list(StratifiedKFold(N_FOLDS, shuffle=True, random_state=SEED)
                 .split(X_train, y_train))
assert set(train_ids).isdisjoint(test_ids)
assert np.array_equal(np.sort(np.concatenate([v for _, v in CV_SPLITS])),
                      np.arange(len(X_train)))

fold_assignment = np.full(len(X_train), -1)
for fold, (_, valid) in enumerate(CV_SPLITS):
    fold_assignment[valid] = fold
split_manifest = pd.concat([
    pd.DataFrame({'row_id': train_ids, 'split': 'train', 'cv_validation_fold': fold_assignment}),
    pd.DataFrame({'row_id': test_ids, 'split': 'test', 'cv_validation_fold': -1}),
], ignore_index=True)
split_manifest.to_csv(RESULTS_DIR / 'split_manifest.csv', index=False)
data_summary = pd.DataFrame([
    {'Partition': label, 'Events': len(target), 'Gamma': int(target.sum()),
     'Hadron': int((1 - target).sum()), 'Gamma fraction': target.mean()}
    for label, target in [('All retained', y), ('Training', y_train), ('Test', y_test)]
])
data_summary.to_csv(RESULTS_DIR / 'data_summary.csv', index=False)
print('Duplicate feature rows removed:', int((~keep).sum()))
show_table(data_summary, 'MAGIC Gamma Telescope: fixed stratified split')

Duplicate feature rows removed: 115
+---------------------------------------------------------+
|      MAGIC Gamma Telescope: fixed stratified split      |
+--------------+--------+-------+--------+----------------+
| Partition    | Events | Gamma | Hadron | Gamma fraction |
+--------------+--------+-------+--------+----------------+
| All retained | 18905  | 12332 | 6573   | 0.6523         |
| Training     | 15124  | 9866  | 5258   | 0.6523         |
| Test         | 3781   | 2466  | 1315   | 0.6522         |
+--------------+--------+-------+--------+----------------+


### Training-data distributions and dependence

EDA uses only the training partition. The table summarizes all ten features;
the box plots compare four image descriptors between classes. Boxes show the
interquartile range and median; whiskers extend to the most extreme observations
within 1.5 IQR. Individual outlier points are omitted from the plot, not from the
data. The summary table retains the full minimum and maximum.

Spearman correlation measures monotonic association between feature ranks.
Correlated descriptors may contain overlapping information, but correlation alone
does not determine their predictive value. All ten features are retained, with
no outlier trimming, imputation, or scaling for the tree-based classifier.

In [4]:
feature_summary = X_train.describe().T[['min', '25%', '50%', '75%', 'max']]
feature_summary = feature_summary.rename(columns={'50%': 'median'})
feature_summary.insert(0, 'missing', X_train.isna().sum())
feature_summary.insert(1, 'unique', X_train.nunique())
feature_summary = feature_summary.rename_axis('Feature').reset_index()
feature_summary.to_csv(RESULTS_DIR / 'eda_feature_summary.csv', index=False)
show_table(feature_summary, 'Training features: missing values, ranges, and quartiles', precision=3)
assert feature_summary['missing'].eq(0).all()
assert feature_summary['unique'].gt(1).all()

EDA_FEATURES = ['fLength', 'fWidth', 'fSize', 'fAlpha']
FEATURE_LABELS = ['Image length (mm)', 'Image width (mm)',
                  'Image size (log10 photon content)', 'Orientation angle (degrees)']
CLASS_NAMES = {1: 'Gamma', 0: 'Hadron'}
CLASS_COLORS = {1: '#0072B2', 0: '#D55E00'}
eda_data = X_train[EDA_FEATURES].assign(Class=y_train.map(CLASS_NAMES))
eda_data.to_csv(RESULTS_DIR / 'eda_distributions.csv', index_label='row_id')
class_medians = eda_data.groupby('Class')[EDA_FEATURES].median().reindex(['Gamma', 'Hadron'])
class_medians.to_csv(RESULTS_DIR / 'eda_class_medians.csv')
show_table(class_medians.reset_index(), 'Training-set medians by class', precision=3)

box_statistics = []
fig = make_subplots(rows=2, cols=2, subplot_titles=EDA_FEATURES,
                    horizontal_spacing=.14, vertical_spacing=.22)
for i, (feature, label) in enumerate(zip(EDA_FEATURES, FEATURE_LABELS)):
    for value in [1, 0]:
        observations = X_train.loc[y_train == value, feature]
        q1, median, q3 = observations.quantile([.25, .5, .75])
        iqr = q3 - q1
        lower = observations[observations >= q1 - 1.5 * iqr].min()
        upper = observations[observations <= q3 + 1.5 * iqr].max()
        assert lower <= q1 <= median <= q3 <= upper
        box_statistics.append({'Feature': feature, 'Class': CLASS_NAMES[value],
                               'q1': q1, 'median': median, 'q3': q3,
                               'lower_whisker': lower, 'upper_whisker': upper})
        fig.add_trace(go.Box(
            x=[CLASS_NAMES[value]], q1=[q1], median=[median], q3=[q3], lowerfence=[lower], upperfence=[upper],
            name=CLASS_NAMES[value], legendgroup=CLASS_NAMES[value],
            showlegend=False, boxpoints=False, quartilemethod='linear',
            marker_color=CLASS_COLORS[value], line=dict(width=2),
        ), row=i // 2 + 1, col=i % 2 + 1)
    fig.update_yaxes(title_text=label, row=i // 2 + 1, col=i % 2 + 1)
fig.update_layout(title='Image descriptors differ between signal and background<br>'
                  f'<sup>Training only: {len(y_train):,} events · Gamma n={int(y_train.sum()):,}'
                  f' · Hadron n={int((1-y_train).sum()):,} · outlier points hidden; no rows removed</sup>')
pd.DataFrame(box_statistics).to_csv(RESULTS_DIR / 'eda_box_statistics.csv', index=False)
show_and_save(fig, '00a_feature_distributions', height=1000)

correlations = X_train.corr(method='spearman')
correlations.to_csv(RESULTS_DIR / 'eda_spearman_correlations.csv', index_label='Feature')
fig = go.Figure(go.Heatmap(
    z=correlations.to_numpy(), x=correlations.columns, y=correlations.index,
    zmin=-1, zmax=1, zmid=0, colorscale='RdBu', reversescale=True,
    text=correlations.round(2).to_numpy(), texttemplate='%{text:.2f}',
    textfont=dict(size=14), colorbar=dict(title='Spearman ρ'),
    hovertemplate='%{y} / %{x}<br>Spearman ρ=%{z:.3f}<extra></extra>',
))
fig.update_xaxes(tickangle=-35, side='bottom')
fig.update_layout(title='Some image descriptors contain strongly related information<br>'
                  '<sup>Training features · Spearman rank correlation · association, not causation</sup>')
show_and_save(fig, '00b_feature_correlations', height=1000)

upper_triangle = correlations.where(np.triu(np.ones(correlations.shape), k=1).astype(bool)).stack()
strongest_pair = upper_triangle.abs().idxmax()
display(Markdown(
    f"**EDA findings.** Gamma events account for {y_train.mean():.1%} of training observations. "
    f"Median fAlpha is {class_medians.loc['Gamma', 'fAlpha']:.2f}° for gamma events and "
    f"{class_medians.loc['Hadron', 'fAlpha']:.2f}° for hadrons. The largest absolute feature "
    f"correlation is between {strongest_pair[0]} and {strongest_pair[1]} "
    f"(Spearman ρ={upper_triangle.loc[strongest_pair]:.3f}). "
    "The different feature ranges and class distributions motivate nonlinear classification; "
    "the overlapping distributions do not establish a perfect single-feature rule."
))

+--------------------------------------------------------------------------------+
|            Training features: missing values, ranges, and quartiles            |
+----------+---------+--------+----------+---------+---------+---------+---------+
| Feature  | missing | unique | min      | 25%     | median  | 75%     | max     |
+----------+---------+--------+----------+---------+---------+---------+---------+
| fLength  | 0       | 14956  | 4.284    | 24.414  | 37.240  | 69.624  | 334.177 |
| fWidth   | 0       | 14659  | 0.000    | 11.917  | 17.150  | 24.726  | 256.382 |
| fSize    | 0       | 6360   | 1.941    | 2.476   | 2.740   | 3.100   | 5.323   |
| fConc    | 0       | 5975   | 0.013    | 0.235   | 0.354   | 0.503   | 0.893   |
| fConc1   | 0       | 4192   | 0.000    | 0.129   | 0.196   | 0.285   | 0.675   |
| fAsym    | 0       | 14983  | -457.916 | -20.696 | 3.958   | 24.147  | 575.241 |
| fM3Long  | 0       | 14982  | -331.780 | -12.713 | 15.338  | 35.715  | 238.321 |
| fM

**EDA findings.** Gamma events account for 65.2% of training observations. Median fAlpha is 9.50° for gamma events and 42.90° for hadrons. The largest absolute feature correlation is between fConc and fConc1 (Spearman ρ=0.986). The different feature ranges and class distributions motivate nonlinear classification; the overlapping distributions do not establish a perfect single-feature rule.

## 2. Model, search space, and objective

Histogram gradient boosting adds decision trees to reduce logistic loss:

$$
F_M(x)=F_0(x)+\eta\sum_{m=1}^{M}h_m(x),\qquad
P(y=1\mid x)=\operatorname{sigmoid}(F_M(x)).
$$

Learning rate and iteration count interact: a small learning rate may need more
trees. Leaf count controls tree complexity; minimum leaf size and L2 regularization
control overfitting. Feature subsampling supplies a sixth tunable dimension.
The fixed search space includes both conservative and flexible models.

For a configuration $\theta$, maximize the mean of three fixed validation AUCs:

$$
f(\theta)=\frac{1}{3}\sum_{k=1}^{3}
\operatorname{ROC\ AUC}(y_k,\widehat p_{\theta,-k}).
$$

ROC AUC measures ranking, not calibration or a chosen decision threshold.
The 20% holdout is used only after all searches and configuration selection.
Every trial trains three fresh models on the same folds. Early stopping is
disabled so the nominated iteration count is actually used. One computational
thread per fit makes overhead and training-time comparisons easier to interpret.

In [5]:
# name: (lower bound, upper bound, type, logarithmic prior)
SPACE = {
    'learning_rate': (0.005, 0.30, 'float', True),
    'max_iter': (40, 300, 'int', False),
    'max_leaf_nodes': (4, 64, 'int', False),
    'min_samples_leaf': (5, 100, 'int', False),
    'l2_regularization': (1e-4, 10.0, 'float', True),
    'max_features': (0.5, 1.0, 'float', False),
}
PARAM_NAMES = list(SPACE)
space_table = pd.DataFrame([
    {'Parameter': name, 'Lower': low, 'Upper': high,
     'Type': kind, 'Prior': 'log-uniform' if log else 'uniform'}
    for name, (low, high, kind, log) in SPACE.items()
])
space_table.to_csv(RESULTS_DIR / 'search_space.csv', index=False)
show_table(space_table, 'Six shared hyperparameter distributions')


def sample_params(rng) -> dict:
    params = {}
    for name, (low, high, kind, log) in SPACE.items():
        if kind == 'int':
            params[name] = int(rng.integers(low, high + 1))
        else:
            params[name] = float(np.exp(rng.uniform(np.log(low), np.log(high)))
                                 if log else rng.uniform(low, high))
    return params


def encode(params_list) -> np.ndarray:
    values = np.array([[p[name] for name in PARAM_NAMES] for p in params_list], dtype=float)
    for j, (low, high, _, log) in enumerate(SPACE.values()):
        if log:
            values[:, j] = (np.log(values[:, j]) - np.log(low)) / np.log(high / low)
        else:
            values[:, j] = (values[:, j] - low) / (high - low)
    return values


def make_model(params=None):
    return HistGradientBoostingClassifier(
        **(params or {}), early_stopping=False, random_state=SEED,
    )


def evaluate(params) -> np.ndarray:
    with threadpool_limits(limits=1):
        return cross_val_score(make_model(params), X_train, y_train,
                               cv=CV_SPLITS, scoring='roc_auc', n_jobs=1,
                               error_score='raise')


baseline_folds = evaluate({})
baseline_cv = float(baseline_folds.mean())
pd.DataFrame({'fold': range(N_FOLDS), 'roc_auc': baseline_folds}).to_csv(
    RESULTS_DIR / 'baseline_cv.csv', index=False)
show_table(pd.DataFrame([{'Model': 'Default HGB (early stopping off)',
                          'CV ROC AUC': baseline_cv,
                          'Fold SD': baseline_folds.std(ddof=1)}]), 'Untuned reference')

+--------------------------------------------------------------+
|           Six shared hyperparameter distributions            |
+-------------------+---------+----------+-------+-------------+
| Parameter         | Lower   | Upper    | Type  | Prior       |
+-------------------+---------+----------+-------+-------------+
| learning_rate     | 0.0050  | 0.3000   | float | log-uniform |
| max_iter          | 40.0000 | 300.0000 | int   | uniform     |
| max_leaf_nodes    | 4.0000  | 64.0000  | int   | uniform     |
| min_samples_leaf  | 5.0000  | 100.0000 | int   | uniform     |
| l2_regularization | 0.0001  | 10.0000  | float | log-uniform |
| max_features      | 0.5000  | 1.0000   | float | uniform     |
+-------------------+---------+----------+-------+-------------+
+---------------------------------------------------------+
|                    Untuned reference                    |
+----------------------------------+------------+---------+
| Model                            | CV 

## 3. Manual Bayesian optimization and random search

### Gaussian-process surrogate

Each evaluated configuration becomes a point in a six-dimensional unit cube.
Log-distributed parameters are log-transformed before normalization. A Gaussian
process with a Matérn-$5/2$ kernel predicts mean $\mu(\theta)$ and standard
deviation $\sigma(\theta)$ for the objective. The kernel length scale is fixed at
0.35 in normalized coordinates; `alpha=1e-4` is regularization on normalized
targets. The kernel hyperparameters are not optimized during the search.

### Expected Improvement (EI)

For incumbent $f^+$ and exploration offset $\xi=0.0001$:

$$
z=\frac{\mu-f^+-\xi}{\sigma},\qquad
\operatorname{EI}=(\mu-f^+-\xi)\Phi(z)+\sigma\phi(z).
$$

The next trial approximately maximizes EI over 4,096 freshly sampled candidates;
already-evaluated settings are excluded. This balances promising predicted scores
with uncertain regions. At zero predictive variance, EI reduces to positive
deterministic improvement, $\max(\mu-f^+-\xi,0)$.
Random search draws from the same distributions but ignores previous scores.

Each method gets **40 trials including eight common initial configurations**.
The three search seeds change proposals, not the data split or model seed.
All four methods evaluate the identical initial configurations within each seed;
their evaluation costs are charged separately. No pruning or cross-method cache
gives one method a larger effective budget. Timing begins before a proposal and
includes surrogate fitting, acquisition scoring, and objective evaluation.

In [ ]:
def expected_improvement(mu, sigma, best, xi=0.0001):
    mu, sigma = np.asarray(mu), np.asarray(sigma)
    improvement = mu - best - xi
    z = improvement / np.maximum(sigma, 1e-12)
    return np.where(sigma > 1e-12,
                    improvement * norm.cdf(z) + sigma * norm.pdf(z),
                    np.maximum(improvement, 0.0))


def propose_bo(params_history, scores, rng):
    seen = {tuple(p[name] for name in PARAM_NAMES) for p in params_history}
    candidates = [sample_params(rng) for _ in range(4096)]
    candidates = [p for p in candidates
                  if tuple(p[name] for name in PARAM_NAMES) not in seen]
    surrogate = GaussianProcessRegressor(
        kernel=Matern(length_scale=0.35, nu=2.5),
        alpha=1e-4, normalize_y=True, optimizer=None,
    )
    with threadpool_limits(limits=1):
        surrogate.fit(encode(params_history), scores)
        mu, sigma = surrogate.predict(encode(candidates), return_std=True)
    return candidates[int(np.argmax(expected_improvement(mu, sigma, max(scores))))]


def trial_record(method, seed, trial, params, folds, objective_s, elapsed_s):
    return {'method': method, 'seed': seed, 'trial': trial,
            **params, 'score': float(np.mean(folds)),
            **{f'fold_{i}': float(v) for i, v in enumerate(folds)},
            'fold_sd': float(np.std(folds, ddof=1)),
            'objective_s': objective_s, 'elapsed_s': elapsed_s}


def run_manual(method, seed, initial):
    rng = np.random.default_rng(seed + 1000)
    records, params_history, scores = [], [], []
    started = perf_counter()
    for trial in range(N_TRIALS):
        if trial < N_INITIAL:
            params = initial[trial].copy()
        elif method == 'Manual BO':
            params = propose_bo(params_history, scores, rng)
        else:
            params = sample_params(rng)
        objective_start = perf_counter()
        folds = evaluate(params)
        objective_s = perf_counter() - objective_start
        records.append(trial_record(method, seed, trial + 1, params, folds,
                                    objective_s, perf_counter() - started))
        params_history.append(params)
        scores.append(float(folds.mean()))
    return pd.DataFrame(records)


## 4. Optuna implementation

Optuna's **Tree-structured Parzen Estimator (TPE)** is a sequential model-based
optimizer, but it is not the GP above. It models the parameter distributions for
good and other trials and proposes settings with a favorable density ratio.
In Optuna 5.0, the default TPE is multivariate for a single objective:
it can model parameter interactions. We use 64 EI candidates and the same eight
startup trials. Optuna RandomSampler is its library-based control.

Both use the identical model, objective, priors, seeds, and budget as the manual
pair. The ask/tell interface lets the same timing boundary cover both library
proposal generation and bookkeeping.

In [7]:
def run_optuna(method, seed, initial):
    sampler = (optuna.samplers.TPESampler(seed=seed + 1000,
                                         n_startup_trials=N_INITIAL,
                                         n_ei_candidates=64)
               if method == 'Optuna TPE'
               else optuna.samplers.RandomSampler(seed=seed + 1000))
    started = perf_counter()
    study = optuna.create_study(direction='maximize', sampler=sampler)
    for params in initial:
        study.enqueue_trial(params)
    records = []
    for number in range(N_TRIALS):
        trial = study.ask()
        params = {}
        for name, (low, high, kind, log) in SPACE.items():
            suggest = trial.suggest_int if kind == 'int' else trial.suggest_float
            params[name] = suggest(name, low, high, log=log)
        objective_start = perf_counter()
        folds = evaluate(params)
        objective_s = perf_counter() - objective_start
        study.tell(trial, float(folds.mean()))
        records.append(trial_record(method, seed, number + 1, params, folds,
                                    objective_s, perf_counter() - started))
    study.trials_dataframe().to_csv(
        RESULTS_DIR / f'{method.lower().replace(" ", "_")}_{seed}_study.csv', index=False)
    return pd.DataFrame(records), study


histories, studies = [], {}
for seed_index, seed in enumerate(SEARCH_SEEDS):
    rng = np.random.default_rng(seed)
    initial = [sample_params(rng) for _ in range(N_INITIAL)]
    # Rotate execution order to reduce a fixed first-method timing advantage.
    for method in METHODS[seed_index:] + METHODS[:seed_index]:
        if method.startswith('Manual'):
            history = run_manual(method, seed, initial)
        else:
            history, studies[(method, seed)] = run_optuna(method, seed, initial)
        history['best_score'] = history.score.cummax()
        history['trial_s'] = history.elapsed_s.diff().fillna(history.elapsed_s.iloc[0])
        history['overhead_s'] = history.trial_s - history.objective_s
        assert (history.overhead_s >= -1e-9).all()
        assert len(history) == N_TRIALS
        assert (history.best_score.diff().dropna() >= 0).all()
        histories.append(history)
        history.to_csv(RESULTS_DIR / f'{method.lower().replace(" ", "_")}_{seed}.csv', index=False)
        print(f'{method:14s} | seed {seed} | best CV AUC {history.score.max():.5f}'
              f' | {history.elapsed_s.iloc[-1]:.1f} s')

history = pd.concat(histories, ignore_index=True).sort_values(['method', 'seed', 'trial'])
history.to_csv(RESULTS_DIR / 'trial_history.csv', index=False)
for seed in SEARCH_SEEDS:
    warmups = [history.query('method == @method and seed == @seed').head(N_INITIAL)
               for method in METHODS]
    for other in warmups[1:]:
        np.testing.assert_allclose(warmups[0][PARAM_NAMES + ['score']],
                                   other[PARAM_NAMES + ['score']], rtol=0, atol=1e-12)
assert len(history) == len(METHODS) * len(SEARCH_SEEDS) * N_TRIALS


Manual BO      | seed 334771 | best CV AUC 0.93191 | 41.0 s
Manual random  | seed 334771 | best CV AUC 0.93148 | 30.8 s
Optuna TPE     | seed 334771 | best CV AUC 0.93243 | 41.9 s
Optuna random  | seed 334771 | best CV AUC 0.93175 | 26.6 s
Manual random  | seed 334772 | best CV AUC 0.93167 | 31.2 s
Optuna TPE     | seed 334772 | best CV AUC 0.93220 | 27.0 s
Optuna random  | seed 334772 | best CV AUC 0.93224 | 32.7 s
Manual BO      | seed 334772 | best CV AUC 0.93285 | 43.4 s
Optuna TPE     | seed 334773 | best CV AUC 0.93225 | 37.8 s
Optuna random  | seed 334773 | best CV AUC 0.93175 | 26.4 s
Manual BO      | seed 334773 | best CV AUC 0.93244 | 50.0 s
Manual random  | seed 334773 | best CV AUC 0.93205 | 27.5 s


## 5. Objective values and convergence

The left panels show **every trial score for the first preselected seed**, with
its best-so-far envelope. A poor trial can be exploration rather than a failure;
the envelope cannot decrease. The right panels summarize all three seeds.
Shading is the **observed minimum–maximum across search seeds**, not a confidence
interval and not variability across dataset splits. The warm-up costs eight of
the 40 trials and is included in every curve.

In [8]:
def convergence_figure(methods, title, stem):
    fig = make_subplots(rows=1, cols=2,
                        subplot_titles=['Individual trials: fixed first seed',
                                        'Best so far: three search seeds'])
    for method in methods:
        group = history[history.method == method]
        first = group[group.seed == SEARCH_SEEDS[0]]
        fig.add_trace(go.Scatter(x=first.trial, y=first.score, mode='markers',
                                name=method, legendgroup=method,
                                marker=dict(color=COLORS[method], symbol=SYMBOLS[method],
                                            size=8, opacity=1.0)), row=1, col=1)
        fig.add_trace(go.Scatter(x=first.trial, y=first.best_score, mode='lines',
                                showlegend=False, legendgroup=method,
                                line=dict(color=COLORS[method], dash=DASHES[method], width=3)),
                      row=1, col=1)
        agg = group.groupby('trial').best_score.agg(['mean', 'min', 'max']).reset_index()
        color = COLORS[method].lstrip('#')
        rgb = ','.join(str(int(color[i:i+2], 16)) for i in (0, 2, 4))
        fig.add_trace(go.Scatter(x=agg.trial, y=agg['min'], mode='lines',
                                line=dict(width=0), showlegend=False, hoverinfo='skip'), row=1, col=2)
        fig.add_trace(go.Scatter(x=agg.trial, y=agg['max'], mode='lines',
                                line=dict(width=0), fill='tonexty', fillcolor=f'rgba({rgb},0.12)',
                                showlegend=False, hoverinfo='skip'), row=1, col=2)
        fig.add_trace(go.Scatter(x=agg.trial, y=agg['mean'], mode='lines', name=method,
                                legendgroup=method, showlegend=False,
                                line=dict(color=COLORS[method], dash=DASHES[method], width=3)),
                      row=1, col=2)
    for col in [1, 2]:
        fig.add_vline(x=N_INITIAL, line_dash='dot', line_color='#8C9298', row=1, col=col)
        fig.add_hline(y=baseline_cv, line_dash='dash', line_color='#737373', row=1, col=col)
    fig.update_xaxes(title_text='Trial (including 8 startup evaluations)', dtick=8)
    fig.update_yaxes(title_text='Validation ROC AUC', row=1, col=1)
    fig.update_yaxes(title_text='Best validation ROC AUC', row=1, col=2)
    fig.update_layout(title=f'{title}<br><sup>Higher is better · bands: seed range (n=3)'
                           f' · gray horizontal line: default model {baseline_cv:.4f}</sup>')
    show_and_save(fig, stem, height=720)


convergence_figure(METHODS[:2], 'Manual GP–EI versus random search', '01_manual_convergence')
convergence_figure(METHODS[2:], 'Optuna TPE versus RandomSampler', '02_optuna_convergence')

## 6. Where did each method search?

PCA projects **hyperparameter settings**, not telescope observations. It is fitted
once to the shared six-dimensional normalized representation from all methods
and seeds; all panels use the same axes and AUC color scale. The diamond marks
the highest-CV setting within each method. Nearby projected points can still be
far apart in dimensions discarded by PCA, so this picture is exploratory rather
than a complete response surface. Explained variance quantifies the information
retained by the projection.

In [15]:
encoded = encode(history[PARAM_NAMES].to_dict('records'))
pca = PCA(n_components=2, svd_solver='full')
coordinates = pca.fit_transform(encoded)
projection = history[['method', 'seed', 'trial', 'score']].copy()
projection[['PC1', 'PC2']] = coordinates
projection.to_csv(RESULTS_DIR / 'search_projection.csv', index=False)
pd.DataFrame(pca.components_.T, index=PARAM_NAMES, columns=['PC1', 'PC2']).to_csv(
    RESULTS_DIR / 'pca_loadings.csv', index_label='parameter')
fig = make_subplots(rows=2, cols=2, subplot_titles=METHODS,
                    horizontal_spacing=0.12, vertical_spacing=0.17)
for i, method in enumerate(METHODS):
    points = projection[projection.method == method]
    row, col = i // 2 + 1, i % 2 + 1
    fig.add_trace(go.Scatter(
        x=points.PC1, y=points.PC2, mode='markers', showlegend=False,
        customdata=points[['seed', 'trial', 'score']],
        hovertemplate='seed %{customdata[0]} · trial %{customdata[1]}<br>AUC %{customdata[2]:.5f}<extra></extra>',
        marker=dict(size=8, color=points.score, coloraxis='coloraxis',
                    line=dict(color='#536170', width=0.5))), row=row, col=col)
    best = points.loc[points.score.idxmax()]
    fig.add_trace(go.Scatter(x=[best.PC1], y=[best.PC2], mode='markers',
                            name='Best CV setting', showlegend=i == 0,
                            marker=dict(symbol='diamond-open', size=19, color='#111111',
                                        line=dict(width=2, color='#111111'))), row=row, col=col)
fig.update_xaxes(title_text=f'PC1 ({pca.explained_variance_ratio_[0]:.1%})',
                 range=[coordinates[:, 0].min() - .1, coordinates[:, 0].max() + .1])
fig.update_yaxes(title_text=f'PC2 ({pca.explained_variance_ratio_[1]:.1%})',
                 range=[coordinates[:, 1].min() - .1, coordinates[:, 1].max() + .1])
fig.update_layout(
    coloraxis=dict(colorscale='Cividis', cmin=history.score.min(), cmax=history.score.max(),
                   colorbar=dict(title='CV AUC', len=.85, thickness=18)),
)
show_and_save(fig, '03_search_space', height=1000)

PosixPath('/Users/lopatenko/Desktop/itmo/itmo-ml-2025/automl/lab-1/figures/03_search_space.png')

## 7. Hyperparameter importance

These are importances of the **six model settings**, not of the ten input features.
For each method, a random-forest response model learns configuration → CV AUC.
We remove repeated settings, use five-fold cross-fitting over configurations, and
compute permutation importance **on unseen configuration folds**, not on the
response model's training rows. The same procedure covers both manual searches
and both Optuna searches.

Importance is the increase in held-out mean squared prediction error after
permuting a parameter. Values near zero mean little measurable predictive
contribution here; negative values are retained. Whiskers show the SD of the
five fold-level means (each averages ten permutations), **not confidence intervals**.
Out-of-fold $R^2$ indicates whether the response model is trustworthy enough to
interpret. Different methods explore different distributions, so importances are
local to their sampled regions and are not causal or globally comparable rankings.

For the Optuna studies we additionally export native fANOVA importance, averaged
over the three study seeds. This normalized in-sample alternative is descriptive;
it must not be confused with held-out permutation importance.

In [10]:
importance_rows, fidelity_rows = [], []
for method in METHODS:
    trials = history[history.method == method].drop_duplicates(PARAM_NAMES)
    Z, scores = encode(trials[PARAM_NAMES].to_dict('records')), trials.score.to_numpy()
    predicted = np.full(len(trials), np.nan)
    fold_importances = []
    for fold, (train, valid) in enumerate(KFold(5, shuffle=True, random_state=SEED).split(Z)):
        response_model = RandomForestRegressor(n_estimators=300, min_samples_leaf=2,
                                                random_state=SEED + fold, n_jobs=1)
        response_model.fit(Z[train], scores[train])
        predicted[valid] = response_model.predict(Z[valid])
        permuted = permutation_importance(response_model, Z[valid], scores[valid],
                                          scoring='neg_mean_squared_error', n_repeats=10,
                                          random_state=SEED + fold, n_jobs=1)
        fold_importances.append(permuted.importances_mean)
    fold_importances = np.asarray(fold_importances)
    fidelity_rows.append({'method': method, 'unique_configs': len(trials),
                          'oof_r2': r2_score(scores, predicted)})
    for j, name in enumerate(PARAM_NAMES):
        importance_rows.append({'method': method, 'parameter': name,
                                 'importance': fold_importances[:, j].mean(),
                                 'fold_sd': fold_importances[:, j].std(ddof=1)})
importance = pd.DataFrame(importance_rows)
fidelity = pd.DataFrame(fidelity_rows)
importance.to_csv(RESULTS_DIR / 'permutation_importance.csv', index=False)
fidelity.to_csv(RESULTS_DIR / 'importance_fidelity.csv', index=False)
show_table(fidelity, 'Cross-fitted response model: interpret importance with this fidelity')

native_rows = []
for (method, seed), study in studies.items():
    native = optuna.importance.get_param_importances(
        study, evaluator=optuna.importance.FanovaImportanceEvaluator(seed=SEED))
    native_rows.extend({'method': method, 'seed': seed, 'parameter': name,
                        'importance': value} for name, value in native.items())
native_importance = pd.DataFrame(native_rows)
native_importance.to_csv(RESULTS_DIR / 'optuna_fanova_importance.csv', index=False)
show_table(native_importance.groupby(['method', 'parameter']).importance.mean().unstack(0)
           .reset_index(), 'Optuna native fANOVA: mean normalized importance over 3 studies')

fig = make_subplots(rows=2, cols=2, subplot_titles=[
    f'{method} · OOF R² {fidelity.set_index("method").loc[method, "oof_r2"]:.2f}'
    for method in METHODS], horizontal_spacing=.22, vertical_spacing=.17)
order = importance.groupby('parameter').importance.mean().sort_values().index.tolist()
for i, method in enumerate(METHODS):
    values = importance[importance.method == method].set_index('parameter').loc[order]
    fig.add_trace(go.Scatter(
        x=values.importance * 1e4, y=values.index, mode='markers', showlegend=False,
        marker=dict(color=COLORS[method], symbol=SYMBOLS[method], size=11),
        error_x=dict(type='data', array=values.fold_sd * 1e4, thickness=1.5)),
        row=i // 2 + 1, col=i % 2 + 1)
    fig.add_vline(x=0, line_color='#9AA2AA', row=i // 2 + 1, col=i % 2 + 1)
fig.update_xaxes(title_text='Increase in held-out MSE × 10⁴')
fig.update_yaxes(categoryorder='array', categoryarray=order)
fig.update_layout(title='Which settings explain performance in the explored regions?<br>'
                        '<sup>Held-out permutation importance · whiskers: fold SD (5 folds × 10 permutations)'
                        ' · panel scales differ</sup>')
show_and_save(fig, '04_hyperparameter_importance', height=1050)

+----------------------------------------------------------------------+
| Cross-fitted response model: interpret importance with this fidelity |
+--------------------------+----------------------------+--------------+
| method                   | unique_configs             | oof_r2       |
+--------------------------+----------------------------+--------------+
| Manual BO                | 120                        | 0.4857       |
| Manual random            | 120                        | 0.7788       |
| Optuna TPE               | 120                        | 0.6115       |
| Optuna random            | 120                        | 0.7450       |
+--------------------------+----------------------------+--------------+
+-----------------------------------------------------------------+
| Optuna native fANOVA: mean normalized importance over 3 studies |
+--------------------------+----------------+---------------------+
| parameter                | Optuna TPE     | Optuna random       

PosixPath('/Users/lopatenko/Desktop/itmo/itmo-ml-2025/automl/lab-1/figures/04_hyperparameter_importance.png')

## 8. Equal-budget results and computational cost

The comparison unit is a **complete 40-trial search**, not an individual fold.
Final best AUC measures the endpoint. Mean incumbent AUC averages the best-so-far
score across all 40 steps, rewarding good solutions found early. Paired BO-minus-
random differences share the initial design within each seed. Three seeds support
a descriptive comparison, not a reliable significance test.

Total elapsed time includes proposals and objective fits but excludes dataset
download, baseline training, plotting, and file export. It measures this CPU and
software environment, not an intrinsic method constant. The random methods may
evaluate cheaper models, so evaluation-count and time efficiency are distinct.

In [11]:
run_summary = history.groupby(['method', 'seed'], sort=False).agg(
    best_cv=('score', 'max'), mean_incumbent=('best_score', 'mean'),
    search_s=('elapsed_s', 'max'), objective_s=('objective_s', 'sum'),
    overhead_s=('overhead_s', 'sum'),
).reset_index()
run_summary['gain_vs_default'] = run_summary.best_cv - baseline_cv
run_summary.to_csv(RESULTS_DIR / 'run_summary.csv', index=False)
summary = run_summary.groupby('method').agg(
    best_cv_mean=('best_cv', 'mean'), best_cv_sd=('best_cv', 'std'),
    mean_incumbent=('mean_incumbent', 'mean'),
    search_s_mean=('search_s', 'mean'), overhead_s_mean=('overhead_s', 'mean'),
).reindex(METHODS).reset_index()
summary.to_csv(RESULTS_DIR / 'search_summary.csv', index=False)
show_table(summary, 'Three 40-trial searches per method: mean and search-seed SD', precision=5)
paired_rows = []
for bo, random in [('Manual BO', 'Manual random'), ('Optuna TPE', 'Optuna random')]:
    left = run_summary[run_summary.method == bo].set_index('seed')
    right = run_summary[run_summary.method == random].set_index('seed')
    for seed in SEARCH_SEEDS:
        paired_rows.append({'comparison': f'{bo} − {random}', 'seed': seed,
                            'best_auc_delta': left.loc[seed, 'best_cv'] - right.loc[seed, 'best_cv'],
                            'mean_incumbent_delta': left.loc[seed, 'mean_incumbent'] - right.loc[seed, 'mean_incumbent']})
paired = pd.DataFrame(paired_rows)
paired.to_csv(RESULTS_DIR / 'paired_differences.csv', index=False)
show_table(paired, 'Paired differences: positive favors the model-based method', precision=5)

fig = make_subplots(rows=1, cols=2,
                    subplot_titles=['Best CV AUC at trial 40: all three seeds',
                                    'Time per 40-trial search: seed mean'])
for method in METHODS:
    values = run_summary[run_summary.method == method]
    position = len(METHODS) - 1 - METHODS.index(method)
    fig.add_trace(go.Scatter(x=values.best_cv, y=position + np.linspace(-.10, .10, len(values)), mode='markers',
                            name=method, showlegend=False,
                            marker=dict(color=COLORS[method], symbol=SYMBOLS[method], size=11),
                            customdata=values.seed,
                            hovertemplate='AUC %{x:.5f} · seed %{customdata}<extra></extra>'), row=1, col=1)
mean_cost = run_summary.groupby('method')[['objective_s', 'overhead_s']].mean().reindex(METHODS)
for column, label, color in [('objective_s', 'Model evaluation', '#536D82'),
                             ('overhead_s', 'Search overhead', '#E69F00')]:
    fig.add_trace(go.Bar(x=mean_cost[column], y=METHODS, name=label, orientation='h',
                        marker_color=color), row=1, col=2)
fig.add_vline(x=baseline_cv, line_dash='dash', line_color='#737373', row=1, col=1)
fig.update_yaxes(tickmode='array', tickvals=[3, 2, 1, 0], ticktext=METHODS,
                 range=[-.5, 3.5], row=1, col=1)
fig.update_yaxes(categoryorder='array', categoryarray=METHODS[::-1], row=1, col=2)
fig.update_xaxes(title_text='Best validation ROC AUC', row=1, col=1)
fig.update_xaxes(title_text='Seconds (includes proposal overhead)', rangemode='tozero', row=1, col=2)
fig.update_layout(barmode='stack', title='Solution quality and runtime answer different questions<br>'
                  '<sup>Equal evaluation budgets · three paired seeds (vertically offset) · dashed line: untuned model</sup>')
show_and_save(fig, '05_quality_and_cost', height=720)

fig = go.Figure()
for method in METHODS:
    runs = [history[(history.method == method) & (history.seed == seed)] for seed in SEARCH_SEEDS]
    start = max(run.elapsed_s.iloc[0] for run in runs)
    stop = min(run.elapsed_s.iloc[-1] for run in runs)
    grid = np.linspace(start, stop, 150)
    incumbents = np.array([run.best_score.to_numpy()[
        np.searchsorted(run.elapsed_s.to_numpy(), grid, side='right') - 1] for run in runs])
    fig.add_trace(go.Scatter(x=grid, y=incumbents.mean(axis=0), mode='lines', name=method,
                            line=dict(color=COLORS[method], dash=DASHES[method], width=3)))
fig.update_layout(title='Best-so-far quality versus elapsed search time<br>'
                  '<sup>Mean of three seeds · each line stops at its shortest completed run'
                  ' · no extrapolation beyond the observed searches</sup>',
                  xaxis_title='Elapsed seconds', yaxis_title='Best validation ROC AUC')
show_and_save(fig, '06_time_convergence', height=700)

+----------------------------------------------------------------------------------------------+
|                 Three 40-trial searches per method: mean and search-seed SD                  |
+---------------+--------------+------------+----------------+---------------+-----------------+
| method        | best_cv_mean | best_cv_sd | mean_incumbent | search_s_mean | overhead_s_mean |
+---------------+--------------+------------+----------------+---------------+-----------------+
| Manual BO     | 0.93240      | 0.00047    | 0.93149        | 44.80040      | 0.89466         |
| Manual random | 0.93173      | 0.00029    | 0.93109        | 29.80877      | 0.00184         |
| Optuna TPE    | 0.93229      | 0.00012    | 0.93142        | 35.58580      | 0.09977         |
| Optuna random | 0.93191      | 0.00028    | 0.93121        | 28.57434      | 0.01067         |
+---------------+--------------+------------+----------------+---------------+-----------------+
+-----------------------------

PosixPath('/Users/lopatenko/Desktop/itmo/itmo-ml-2025/automl/lab-1/figures/06_time_convergence.png')

## 9. Untouched test set

One champion per method is selected by **CV score only** across its three
40-trial searches (120 evaluations per method in total), then refitted on all
training rows. This is a pooled-budget diagnostic; do not mistake these champions
for a typical single 40-trial run. The default model is refitted too.

All five models predict the same held-out events. ROC curves visualize the
ranking trade-off; standardized partial AUC focuses on FPR ≤ 0.10. The 95%
stratified bootstrap intervals resample test events 1,000 times using shared
draws across models. They describe finite-test-set uncertainty conditional on the
fitted models, **not** model-training or search uncertainty. Paired bootstrap
differences are reported separately. No settings are revised from these scores.

In [12]:
champions = history.loc[history.groupby('method').score.idxmax()].copy()
champions.to_csv(RESULTS_DIR / 'selected_configurations.csv', index=False)
show_table(champions[['method', 'seed', 'trial', 'score']].sort_values('method'),
           'CV-selected champions (test scores were not used)')
show_table(champions.set_index('method').reindex(METHODS)[PARAM_NAMES].T
           .rename_axis('Parameter').reset_index(),
           'Selected hyperparameters', precision=5)
predictions = {'row_id': test_ids, 'y_true': y_test.to_numpy()}
for method in ['Default HGB'] + METHODS:
    params = {} if method == 'Default HGB' else champions.set_index('method').loc[method, PARAM_NAMES].to_dict()
    for name, (_, _, kind, _) in SPACE.items():
        if name in params and kind == 'int':
            params[name] = int(params[name])
    with threadpool_limits(limits=1):
        fitted = make_model(params).fit(X_train, y_train)
        predictions[method] = fitted.predict_proba(X_test)[:, 1]
predictions = pd.DataFrame(predictions)
predictions.to_csv(RESULTS_DIR / 'test_predictions.csv', index=False)
rng = np.random.default_rng(SEED)
labels = y_test.to_numpy()
class_indices = [np.flatnonzero(labels == value) for value in [0, 1]]
test_methods = ['Default HGB'] + METHODS
boot_auc = np.empty((1000, len(test_methods)))
for b in range(len(boot_auc)):
    indices = np.concatenate([rng.choice(group, len(group), replace=True) for group in class_indices])
    for j, method in enumerate(test_methods):
        boot_auc[b, j] = roc_auc_score(labels[indices], predictions[method].to_numpy()[indices])
test_rows = []
roc_rows = []
fig = make_subplots(rows=1, cols=2, subplot_titles=['Full ROC', 'Low false-positive region'])
for j, method in enumerate(test_methods):
    probability = predictions[method].to_numpy()
    low, high = np.quantile(boot_auc[:, j], [.025, .975])
    test_rows.append({'method': method, 'roc_auc': roc_auc_score(labels, probability),
                     'ci_low': low, 'ci_high': high,
                     'partial_auc_fpr_0.10': roc_auc_score(labels, probability, max_fpr=.10)})
    fpr, tpr, thresholds = roc_curve(labels, probability)
    roc_rows.extend({'method': method, 'fpr': a, 'tpr': b, 'threshold': t}
                    for a, b, t in zip(fpr, tpr, thresholds))
    for col in [1, 2]:
        fig.add_trace(go.Scatter(x=fpr, y=tpr, mode='lines', name=method, showlegend=col == 1,
                                line=dict(color=COLORS.get(method, '#777777'),
                                          dash=DASHES.get(method, 'longdash'), width=2.5)), row=1, col=col)
test_results = pd.DataFrame(test_rows)
test_results.to_csv(RESULTS_DIR / 'test_metrics.csv', index=False)
pd.DataFrame(roc_rows).to_csv(RESULTS_DIR / 'test_roc_curves.csv', index=False)
pd.DataFrame(boot_auc, columns=test_methods).to_csv(RESULTS_DIR / 'test_bootstrap_auc.csv', index=False)
show_table(test_results, 'Held-out performance: 95% stratified bootstrap intervals')
test_pair_rows = []
for bo, random in [('Manual BO', 'Manual random'), ('Optuna TPE', 'Optuna random')]:
    delta = boot_auc[:, test_methods.index(bo)] - boot_auc[:, test_methods.index(random)]
    point = test_results.set_index('method').loc[bo, 'roc_auc'] - test_results.set_index('method').loc[random, 'roc_auc']
    test_pair_rows.append({'comparison': f'{bo} − {random}', 'auc_delta': point,
                           'ci_low': np.quantile(delta, .025), 'ci_high': np.quantile(delta, .975)})
test_pairs = pd.DataFrame(test_pair_rows)
test_pairs.to_csv(RESULTS_DIR / 'test_paired_differences.csv', index=False)
show_table(test_pairs, 'Paired test AUC differences: same bootstrap draws', precision=5)
fig.update_xaxes(title_text='False-positive rate', range=[0, 1], row=1, col=1)
fig.update_xaxes(title_text='False-positive rate', range=[0, .10], row=1, col=2)
fig.update_yaxes(title_text='True-positive rate', range=[0, 1], ticklabelstandoff=8)
fig.update_xaxes(ticklabelstandoff=8)
fig.update_layout(title='Generalization on the untouched test events<br>'
                  f'<sup>{len(y_test):,} events · champions selected only by CV · '
                  'uncertainty and paired differences reported in the tables</sup>')
show_and_save(fig, '07_test_roc', height=740)

fig = go.Figure(go.Scatter(
    x=test_results.roc_auc, y=test_results.method, mode='markers',
    marker=dict(size=11, color=[COLORS.get(m, '#777777') for m in test_results.method]),
    error_x=dict(type='data', symmetric=False,
                 array=test_results.ci_high - test_results.roc_auc,
                 arrayminus=test_results.roc_auc - test_results.ci_low),
))
fig.update_layout(title='Small test-score gaps must be read with their uncertainty<br>'
                  '<sup>95% stratified bootstrap intervals · 1,000 resamples · '
                  'conditional on these fitted models</sup>',
                  xaxis_title='Test ROC AUC', yaxis=dict(categoryorder='array',
                                                        categoryarray=test_methods[::-1]), showlegend=False)
show_and_save(fig, '08_test_uncertainty', height=700)

+---------------------------------------------------+
| CV-selected champions (test scores were not used) |
+-------------------+----------+--------+-----------+
| method            | seed     | trial  | score     |
+-------------------+----------+--------+-----------+
| Manual BO         | 334772   | 31     | 0.9328    |
| Manual random     | 334773   | 28     | 0.9321    |
| Optuna TPE        | 334771   | 29     | 0.9324    |
| Optuna random     | 334772   | 9      | 0.9322    |
+-------------------+----------+--------+-----------+
+----------------------------------------------------------------------------+
|                          Selected hyperparameters                          |
+-------------------+-----------+---------------+------------+---------------+
| Parameter         | Manual BO | Manual random | Optuna TPE | Optuna random |
+-------------------+-----------+---------------+------------+---------------+
| learning_rate     | 0.06099   | 0.04857       | 0.04205    | 0.

PosixPath('/Users/lopatenko/Desktop/itmo/itmo-ml-2025/automl/lab-1/figures/08_test_uncertainty.png')

In [14]:
lines = ['### Observed results', '']
for _, row in summary.iterrows():
    lines.append(f"- **{row['method']}**: final best CV AUC "
                 f"{row.best_cv_mean:.5f} ± {row.best_cv_sd:.5f} (search-seed SD), "
                 f"gain {row.best_cv_mean - baseline_cv:+.5f} over default; "
                 f"mean search time {row.search_s_mean:.1f} s.")
lines += ['', '### Paired comparison', '']
for comparison, values in paired.groupby('comparison', sort=False):
    wins = int((values.best_auc_delta > 0).sum())
    lines.append(f'- **{comparison}**: mean final-AUC difference '
                 f'{values.best_auc_delta.mean():+.5f}; model-based search wins {wins}/3 seeds. '
                 f'Mean-incumbent difference: {values.mean_incumbent_delta.mean():+.5f}.')
for _, row in test_pairs.iterrows():
    conclusion = ('contains zero: no clear test-set advantage' if row.ci_low <= 0 <= row.ci_high
                  else 'excludes zero in this conditional test-set comparison')
    lines.append(f'- Test **{row.comparison}**: {row.auc_delta:+.5f}, '
                 f'95% paired interval [{row.ci_low:+.5f}, {row.ci_high:+.5f}] — {conclusion}.')
report = '\n'.join(lines)
display(Markdown(report))
_ = (RESULTS_DIR / 'observed_results.md').write_text(report + '\n', encoding='utf-8')

### Observed results

- **Manual BO**: final best CV AUC 0.93240 ± 0.00047 (search-seed SD), gain +0.00129 over default; mean search time 44.8 s.
- **Manual random**: final best CV AUC 0.93173 ± 0.00029 (search-seed SD), gain +0.00063 over default; mean search time 29.8 s.
- **Optuna TPE**: final best CV AUC 0.93229 ± 0.00012 (search-seed SD), gain +0.00119 over default; mean search time 35.6 s.
- **Optuna random**: final best CV AUC 0.93191 ± 0.00028 (search-seed SD), gain +0.00081 over default; mean search time 28.6 s.

### Paired comparison

- **Manual BO − Manual random**: mean final-AUC difference +0.00066; model-based search wins 3/3 seeds. Mean-incumbent difference: +0.00041.
- **Optuna TPE − Optuna random**: mean final-AUC difference +0.00038; model-based search wins 2/3 seeds. Mean-incumbent difference: +0.00021.
- Test **Manual BO − Manual random**: +0.00130, 95% paired interval [-0.00030, +0.00290] — contains zero: no clear test-set advantage.
- Test **Optuna TPE − Optuna random**: +0.00022, 95% paired interval [-0.00181, +0.00209] — contains zero: no clear test-set advantage.

## References

1. R. Bock. [MAGIC Gamma Telescope](https://doi.org/10.24432/C52C8B), UCI Machine Learning Repository; [OpenML 1120](https://www.openml.org/search?type=data&id=1120), version 1.
2. Scikit-learn. [Histogram gradient boosting](https://scikit-learn.org/1.8/modules/generated/sklearn.ensemble.HistGradientBoostingClassifier.html), [Gaussian-process regression](https://scikit-learn.org/1.8/modules/gaussian_process.html), and [permutation importance](https://scikit-learn.org/1.8/modules/permutation_importance.html).
3. Optuna. [TPESampler](https://optuna.readthedocs.io/en/stable/reference/samplers/generated/optuna.samplers.TPESampler.html) and [hyperparameter importance](https://optuna.readthedocs.io/en/stable/reference/importance.html).
4. Bergstra et al. [Algorithms for Hyper-Parameter Optimization](https://papers.nips.cc/paper/4443-algorithms-for-hyper-parameter-optimization), NeurIPS 2011.